In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
print('Mounting Google Drive...')
drive.mount('/content/drive')

# Create a directory for our data
!mkdir -p '/content/drive/MyDrive/isazi_recruitment'
DB_PATH = '/content/drive/MyDrive/isazi_recruitment/recruitment.db'
print(f'Database will be stored at: {DB_PATH}')

# Lightweight Colab: Run ISAZI Recruitment Dashboard (fast)

This lightweight notebook installs only the commonly required libraries and skips heavy packages like `prophet`, `shap`, and `optuna` to speed up the Colab run. Use this for quick verification and running the Streamlit dashboard.

Notes:
- If you later need heavy libraries, use the full notebook `colab_run_dashboard.ipynb`.
- For a private repo, provide a token or upload the repo zip.

In [ ]:
# 1) Clone the repository (or pull if already present).
import os
os.chdir('/content')
repo_url = 'https://github.com/olwethusibisi7/dataset.git'
if not os.path.exists('dataset'):
    !git clone {repo_url} dataset
else:
    print('dataset already exists; pulling latest')
    os.chdir('dataset')
    !git pull || true
    os.chdir('..')

In [ ]:
# 2) Install lightweight Python packages (fast).
# Skips prophet, shap, optuna to reduce install time.
!python -m pip install --upgrade pip setuptools wheel -q
!pip install -q streamlit pyngrok pytest pandas numpy scikit-learn plotly xgboost
print('Lightweight install finished')

If you need to add heavy libs later, run `!pip install prophet shap optuna` in a separate cell.
Some of those packages can take several minutes or require extra system packages.

In [ ]:
# 3) Run the repository tests (targeted).
import os, sys
os.chdir('/content/dataset')
sys.path.insert(0, os.getcwd())
print('Running tests in "ISAZI DASH/tests" (if present) and ./tests')
# Run the ISAZI DASH tests (path contains a space) then fallback to regular tests
!pytest -q "ISAZI DASH/tests" || true
!pytest -q tests || true

Create sample database rows so the dashboard has data to display. The project exposes `create_sample_data()` in `recruitment_functions.py`.

In [ ]:
# 4) Create sample DB data
import os, sys
os.chdir('/content/dataset')
sys.path.insert(0, os.getcwd())
try:
    from recruitment_functions import create_sample_data, load_recruitment_data
    create_sample_data(force=True)
    df = load_recruitment_data()
    print('Loaded rows:', len(df))
except Exception as e:
    print('Error creating or loading sample data:', e)

In [ ]:
# 5) Start Streamlit and expose via ngrok; prints public URL.
import os, subprocess, time
from pyngrok import ngrok
REPO_DIR = '/content/dataset'
PORT = 8501
os.chdir(REPO_DIR)
print('Starting ngrok tunnel...')
tunnel = ngrok.connect(PORT)
print('Public URL:', tunnel.public_url)
logfile = os.path.join(REPO_DIR, 'streamlit_colab_light.log')
cmd = f'nohup streamlit run dashboard.py --server.port {PORT} --server.headless true > {logfile} 2>&1 &'
print('Starting Streamlit...')
subprocess.call(cmd, shell=True)
time.sleep(3)
print('Streamlit logs (tail):')
!tail -n 40 /content/dataset/streamlit_colab_light.log || true
print('\nOpen the public URL above in your browser to view the dashboard.')

In [ ]:
# OPTIONAL: Import data from Google Sheets (public / published as CSV)
# Paste your Sheet ID below (the long id in the sheet URL) and set GID if needed.
SHEET_ID = ''  # e.g. '1A2b3C4d5EfG...'
GID = 0

if SHEET_ID:
    import pandas as pd
    csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}'
    print('Fetching:', csv_url)
    try:
        df_sheet = pd.read_csv(csv_url)
        print('Imported rows:', len(df_sheet))
        display(df_sheet.head())
        # Save to CSV in the notebook workspace
        df_sheet.to_csv('imported_sheet.csv', index=False)
        print('Saved as imported_sheet.csv')
        # Try to save to the project's SQLite DB (if present)
        try:
            import sqlite3
            conn = sqlite3.connect('recruitment.db')
            df_sheet.to_sql('imported_sheet', conn, if_exists='replace', index=False)
            conn.close()
            print('Saved sheet to recruitment.db -> table imported_sheet')
        except Exception as e:
            print('Could not write to recruitment.db:', e)
    except Exception as e:
        print('Failed to import sheet:', e)
else:
    print('No SHEET_ID provided. To import a Google Sheet, set SHEET_ID variable in this cell.')

Troubleshooting tips:
- If you see import errors, check the pip install output from the earlier cell.
- If Streamlit doesn't start, inspect the `streamlit_colab_light.log` file (last cell's tail output).
- To persist the SQLite DB between Colab sessions, mount Google Drive and write the DB under `/content/drive/MyDrive/...`.